In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

In [2]:
(x_train,y_train),(x_test,y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train.reshape(-1,784)/255.0
x_test = x_test.reshape(-1,784)/255.0

In [3]:
model = Sequential([
    Input(shape=(784,)),
    Dense(128,activation='relu'),
    Dense(64,activation='relu'),
    Dense(10,activation='softmax')
])

In [4]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [5]:
model.fit(x_train,y_train,epochs=5,batch_size=128)

Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.9091 - loss: 0.3208
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9603 - loss: 0.1346
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9713 - loss: 0.0958
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9786 - loss: 0.0710
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9826 - loss: 0.0566


In [6]:
loss,acc = model.evaluate(x_test,y_test)
print("Test Accuracy:",acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9767 - loss: 0.0769
Test Accuracy: 0.9767000079154968


In [7]:
model.save('digit_model.keras')

In [8]:
!pip install flask

In [13]:
from flask import Flask, request, jsonify
import threading

app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    img = np.array(request.json['image']).reshape(1,784)
    pred = int(np.argmax(model.predict(img, verbose=0)))
    return jsonify({'predicted_digit': pred})

threading.Thread(target=lambda: app.run(port=5000, use_reloader=False)).start()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [06/Nov/2026 00:31:45] "POST /predict HTTP/1.1" 200 -


In [14]:
import requests

img = (x_test[0]*255).astype(int).tolist()
r = requests.post('http://127.0.0.1:5000/predict', json={'image':img})
print(r.json())

{'predicted_digit': 7}
